# GNN-RWKV v2.1: Adaptive Social Brain Trainer
This notebook implements the **Adaptive Social Brain** architecture with Context Fusion. 
Designed for deep Hinglish conversational intelligence.

In [ ]:
# 1. Install Dependencies
!pip install datasets torch re gradio

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import json
import re
import collections
from datasets import load_dataset
from google.colab import files
import gradio as gr

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Model Architecture (Adaptive GNN-RWKV)

In [ ]:
class GNNRWKVBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
        self.ln1 = nn.LayerNorm(dim)
        self.ln2 = nn.LayerNorm(dim)
        self.w_receptance = nn.Linear(dim, dim)
        self.w_key = nn.Linear(dim, dim)
        self.w_value = nn.Linear(dim, dim)
        self.word_context_gate = nn.Linear(dim, dim)
        self.time_decay = nn.Parameter(torch.ones(dim) * -0.5)
        self.out_proj = nn.Linear(dim, dim)

    def forward(self, n_x, w_x, edge_index, h):
        context = torch.mean(w_x, dim=0, keepdim=True)
        context_bias = torch.tanh(self.word_context_gate(context))
        n_x_ctx = self.ln1(n_x + context_bias)
        
        r = torch.sigmoid(self.w_receptance(n_x_ctx))
        k = self.w_key(n_x_ctx)
        v = self.w_value(n_x_ctx)
        
        row, col = edge_index
        interaction = torch.zeros_like(k)
        if edge_index.shape[1] > 0:
            interaction.index_add_(0, row, k[col] * v[col])
            
        new_h = (torch.exp(self.time_decay) * h) + interaction
        out = r * self.ln2(new_h)
        return n_x + self.out_proj(out), new_h

class AdaptiveSocialBrain(nn.Module):
    def __init__(self, vocab_size, num_nodes, dim=256, num_layers=4):
        super().__init__()
        self.dim = dim
        self.num_layers = num_layers
        self.word_emb = nn.Embedding(vocab_size, dim)
        self.node_emb = nn.Embedding(num_nodes, dim)
        self.blocks = nn.ModuleList([GNNRWKVBlock(dim) for _ in range(num_layers)])
        self.final_ln = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, vocab_size)

    def forward(self, word_ids, node_ids, edge_index, h_states):
        w_x = self.word_emb(word_ids)
        n_x = self.node_emb(node_ids)
        new_h_states = []
        for i, block in enumerate(self.blocks):
            n_x, new_h = block(n_x, w_x, edge_index, h_states[i])
            new_h_states.append(new_h)
        world_state = n_x.mean(dim=0, keepdim=True)
        combined = w_x + world_state
        logits = self.head(self.final_ln(combined))
        return logits, new_h_states

## 2. Data Preparation

In [ ]:
print("Fetching dataset...")
ds = load_dataset('Abhishekcr448/Hinglish-Everyday-Conversations-1M', split='train[:50000]')

all_text = ""
for row in ds:
    all_text += row['input'] + " " + row['output'] + " "

all_text = all_text.lower().replace(".", " . ").replace(",", " , ")
words = all_text.split()
vocab = sorted(list(set(words)))
w2i = {w: i for i, w in enumerate(vocab)}
i2w = {i: w for i, w in enumerate(vocab)}
vocab_size = len(vocab)
data = [w2i[w] for w in words]

print(f"Vocab Size: {vocab_size} | Total Tokens: {len(data)}")

## 3. Training Loop

In [ ]:
dim = 256 # Higher capacity
num_nodes = 5
num_layers = 4
epochs = 50
chunk_size = 500

model = AdaptiveSocialBrain(vocab_size, num_nodes, dim, num_layers).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
edges = torch.tensor([[0, 1, 1, 3], [1, 0, 3, 1]], dtype=torch.long).to(device)
node_ids = torch.tensor([0, 1, 2, 3, 4]).to(device)

print("Starting Training...")
model.train()
for epoch in range(epochs):
    epoch_loss = 0
    chunks = 0
    for i in range(0, len(data) - chunk_size, chunk_size):
        h_states = [torch.zeros((num_nodes, dim)).to(device) for _ in range(num_layers)]
        optimizer.zero_grad()
        inp = torch.tensor(data[i:i+chunk_size]).to(device)
        tar = torch.tensor(data[i+1:i+chunk_size+1]).to(device)
        logits, _ = model(inp, node_ids, edges, h_states)
        loss = F.cross_entropy(logits, tar)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        chunks += 1
    
    print(f"Epoch {epoch+1}/{epochs} | Avg Loss: {epoch_loss/chunks:.4f}")

print("Training Complete!")

## 4. Gradio Interface (Chat with Brain)

In [ ]:
def chat_with_brain(message, history, temp=0.7, top_p=0.95):
    model.eval()
    clean_input = re.sub(r'[^\w\s]', '', message.lower())
    tokens = clean_input.split()
    input_ids = [w2i[w] for w in tokens if w in w2i]
    
    if not input_ids: return "Mujhe ye samajh nahi aaya."
    
    h = [torch.zeros((num_nodes, dim)).to(device) for _ in range(num_layers)]
    # Build context
    for idx in input_ids:
        with torch.no_grad():
            _, h = model(torch.tensor([idx]).to(device), node_ids, edges, h)
    
    current_id = input_ids[-1]
    response = []
    for _ in range(30):
        with torch.no_grad():
            logits, h = model(torch.tensor([current_id]).to(device), node_ids, edges, h)
            logits = logits[0] / temp
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, 1).item()
            word = i2w[next_id]
            if word == ".": break
            response.append(word)
            current_id = next_id
            
    return " ".join(response)

gr.ChatInterface(chat_with_brain).launch(debug=True)

## 5. Export Weights & Vocab

In [ ]:
torch.save(model.cpu().state_dict(), 'social_brain.pth')
vocab_data = {"w2i": w2i, "i2w": {str(k): v for k, v in i2w.items()}}
with open('vocab.json', 'w') as f:
    json.dump(vocab_data, f)
files.download('social_brain.pth')
files.download('vocab.json')